# 新Patient预测 - KAN & DeepMLP

本notebook对新patient数据使用训练好的KAN和DeepMLP模型进行预测。

## 功能
1. 加载新patient数据（特征+标签）
2. 标签映射（FreeSurfer → 0-51连续索引）
3. 使用KAN和DeepMLP模型预测
4. 保存3D softmax预测结果
5. 生成error map可视化
6. 诊断工具（标签映射验证、数据标准化检查）

## 使用说明
- **正常流程**: 依次运行 Cell 1-6 (设置) → Cell 7-8 (KAN预测+可视化) → Cell 9-10 (DeepMLP预测+可视化)
- **如果准确率很低**: 运行诊断cells (Cell 11和12)检查问题

---

## 📦 Part 1: 导入和配置

In [ ]:
# Cell 1: 导入库和全局配置

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import json
import sys
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ 使用设备: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# FreeSurfer标准标签（52个：0-51）
STANDARD_LABELS = [
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17,
    29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
    44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58,
    59, 60, 61, 62, 103
]

# 创建标签映射
def create_label_mapping():
    """创建FreeSurfer标签到连续索引的映射"""
    forward_mapping = {original: continuous for continuous, original in enumerate(STANDARD_LABELS)}
    reverse_mapping = {continuous: original for continuous, original in enumerate(STANDARD_LABELS)}
    return forward_mapping, reverse_mapping

forward_mapping, reverse_mapping = create_label_mapping()
print(f"\n📋 标签映射: {len(STANDARD_LABELS)}个类别 (0-51)")
print(f"   示例: FreeSurfer {STANDARD_LABELS[:5]} -> 连续索引 [0,1,2,3,4]")

print("\n✅ 导入完成！")

In [ ]:
# Cell 2: 路径配置

# =============================================================================
# 配置区域 - 根据实际情况修改
# =============================================================================

# 服务器路径配置
SERVER_BASE = '/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/D_proj_analysis/banlanced_sample'

# 模型路径配置
MODELS = {
    'KAN': {
        'model_file': f'{SERVER_BASE}/refactored_training/training_runs/kan_bg_excl_20251103_162620/results/kan_bg_excl_20251103_164641.pth',
        'output_dir': f'{SERVER_BASE}/refactored_training/training_runs/kan_bg_excl_20251103_162620/results',
        'model_type': 'kan'
    },
    'DeepMLP': {
        'model_file': f'{SERVER_BASE}/refactored_training/training_runs/deep_mlp_bg_excl_20251103_162620/results/deep_mlp_bg_excl_20251103_200903.pth',
        'output_dir': f'{SERVER_BASE}/refactored_training/training_runs/deep_mlp_bg_excl_20251103_162620/results',
        'model_type': 'deep_mlp'
    }
}

# 新patient数据路径
NEW_PATIENT_DATA = {
    'features': '/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS/FOR_016_20250204_reproducibility/evaluated/stacked_zscore_float32.nii.gz',
    'labels': '/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS/FOR_016_20250204_reproducibility/evaluated/resampled_synthseg_t1_mp2rage_alex_labels.nii.gz',
    'subject_id': 'FOR_016_20250204_reproducibility'
}

# 预测参数
PREDICTION_CONFIG = {
    'batch_size': 8192,
    'exclude_features': [14],  # 排除feature 14
    'include_background': False,  # 排除背景
    'device': device
}

# 验证路径
print("📁 路径配置验证:")
print("\n模型文件:")
for model_name, config in MODELS.items():
    model_path = Path(config['model_file'])
    if model_path.exists():
        size_mb = model_path.stat().st_size / (1024**2)
        print(f"  ✅ {model_name}: {model_path.name} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {model_name}: 文件未找到!")

print("\n新patient数据:")
for data_type, path in NEW_PATIENT_DATA.items():
    if data_type == 'subject_id':
        continue
    data_path = Path(path)
    if data_path.exists():
        size_mb = data_path.stat().st_size / (1024**2)
        print(f"  ✅ {data_type}: {data_path.name} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {data_type}: 文件未找到!")

print(f"\n⚙️ 预测配置:")
print(f"  批大小: {PREDICTION_CONFIG['batch_size']}")
print(f"  排除特征: {PREDICTION_CONFIG['exclude_features']}")
print(f"  排除背景: {PREDICTION_CONFIG['include_background'] == False}")

print("\n✅ 配置完成！")

## 🔧 Part 2: 函数定义

In [ ]:
# Cell 3: 数据加载函数

def load_new_patient_data(feature_path, label_path, subject_id,
                          forward_mapping, include_background=False,
                          exclude_features=None):
    """
    加载新patient数据并进行预处理
    
    关键步骤:
    1. 加载4D特征和3D标签
    2. 标签映射: FreeSurfer → 0-51连续索引
    3. 排除背景体素
    4. 排除指定特征
    
    Returns:
    --------
    dict : 包含处理后的数据和元信息
    """
    print(f"\n📂 加载新patient数据: {subject_id}")
    
    # 1. 加载特征数据
    print(f"  加载特征: {Path(feature_path).name}")
    feature_img = nib.load(feature_path)
    features_4d = feature_img.get_fdata().astype(np.float32)
    print(f"    形状: {features_4d.shape}")
    print(f"    范围: [{features_4d.min():.3f}, {features_4d.max():.3f}]")
    
    # 2. 加载标签数据
    print(f"  加载标签: {Path(label_path).name}")
    label_img = nib.load(label_path)
    labels_3d = label_img.get_fdata().astype(np.int32)
    print(f"    形状: {labels_3d.shape}")
    
    original_shape_3d = labels_3d.shape
    assert features_4d.shape[:3] == labels_3d.shape, "特征和标签的3D形状不匹配！"
    
    # 3. 展平数据
    n_voxels = np.prod(labels_3d.shape)
    n_modalities = features_4d.shape[3]
    
    features_flat = features_4d.reshape(n_voxels, n_modalities)
    labels_flat = labels_3d.flatten()
    
    print(f"  展平: {n_voxels:,} 体素 × {n_modalities} 特征")
    
    # 4. 标签映射 (FreeSurfer → 0-51)
    print(f"  标签映射 (FreeSurfer → 0-51)...")
    unique_orig = np.unique(labels_flat)
    print(f"    原始标签: {len(unique_orig)} 个唯一值")
    
    labels_mapped = np.zeros_like(labels_flat)
    unmapped_labels = []
    
    for orig_label in unique_orig:
        if orig_label in forward_mapping:
            mask = labels_flat == orig_label
            labels_mapped[mask] = forward_mapping[orig_label]
        else:
            unmapped_labels.append(orig_label)
    
    if unmapped_labels:
        print(f"    ⚠️ 未映射的标签: {unmapped_labels}")
    
    labels_flat = labels_mapped
    print(f"    映射后范围: [{labels_flat.min()}, {labels_flat.max()}]")
    
    # 5. 处理背景
    if not include_background:
        spatial_mask_flat = labels_flat != 0
        spatial_mask_3d = spatial_mask_flat.reshape(original_shape_3d)
        flat_indices = np.where(spatial_mask_flat)[0]
        
        features_flat = features_flat[spatial_mask_flat]
        labels_flat = labels_flat[spatial_mask_flat]
        
        print(f"  排除背景: {len(features_flat):,} / {n_voxels:,} 体素 "
              f"({len(features_flat)/n_voxels*100:.2f}%)")
    else:
        spatial_mask_3d = np.ones(original_shape_3d, dtype=bool)
        flat_indices = np.arange(n_voxels)
    
    # 6. 排除指定特征
    if exclude_features:
        keep_mask = np.ones(features_flat.shape[1], dtype=bool)
        keep_mask[exclude_features] = False
        features_flat = features_flat[:, keep_mask]
        print(f"  排除特征{exclude_features}: 保留 {features_flat.shape[1]} 个特征")
    
    # 7. 统计信息
    print(f"\n  📊 数据统计:")
    print(f"    最终形状: {features_flat.shape}")
    print(f"    特征范围: [{features_flat.min():.3f}, {features_flat.max():.3f}]")
    print(f"    特征均值: {features_flat.mean():.3f}")
    print(f"    特征标准差: {features_flat.std():.3f}")
    print(f"    标签范围: [{labels_flat.min()}, {labels_flat.max()}]")
    print(f"    唯一标签数: {len(np.unique(labels_flat))}")
    
    return {
        'features': features_flat,
        'labels': labels_flat,
        'subject_id': subject_id,
        'n_voxels': len(features_flat),
        'original_shape_3d': original_shape_3d,
        'spatial_mask_3d': spatial_mask_3d,
        'flat_indices': flat_indices,
        'affine': feature_img.affine,
        'header': feature_img.header
    }

print("✅ 数据加载函数定义完成！")

In [ ]:
# Cell 4: 模型加载和预测函数

# 添加refactored_training目录到路径
refactored_dir = Path('/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/D_proj_analysis/banlanced_sample/refactored_training')
if str(refactored_dir) not in sys.path:
    sys.path.insert(0, str(refactored_dir))

from models import get_model

def load_trained_model(model_path, model_type, device):
    """
    加载训练好的模型
    
    Returns:
    --------
    model : nn.Module
    checkpoint : dict
    """
    print(f"\n🔧 加载模型: {model_type}")
    print(f"  文件: {Path(model_path).name}")
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    config = checkpoint.get('config', {})
    input_dim = config.get('input_dim', 41)
    num_classes = config.get('num_classes', 52)
    
    print(f"  配置: input_dim={input_dim}, num_classes={num_classes}")
    
    # 创建模型实例（get_model返回(model, config)元组）
    if model_type == 'kan':
        model, _ = get_model('kan', input_dim=input_dim, num_classes=num_classes,
                           grid_size=config.get('grid_size', 8))
    elif model_type == 'deep_mlp':
        model, _ = get_model('deep_mlp', input_dim=input_dim, num_classes=num_classes)
    else:
        raise ValueError(f"不支持的模型类型: {model_type}")
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  参数量: {total_params:,}")
    print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
    
    history = checkpoint.get('history', {})
    if history and 'test_f1' in history:
        test_f1 = history['test_f1']
        if test_f1:
            print(f"  训练时Test F1: {test_f1[-1]:.4f} (历史记录，非当前预测)")
    
    print(f"  ✅ 模型加载完成！")
    return model, checkpoint

def predict_patient(model, data_info, device, batch_size=8192):
    """
    对patient数据进行预测
    
    Returns:
    --------
    predictions : np.ndarray (n_voxels, n_classes)
        Softmax概率分布
    """
    print(f"\n🔮 开始预测...")
    
    features = data_info['features']
    n_samples = len(features)
    
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for start_idx in tqdm(range(0, n_samples, batch_size), desc="预测进度"):
            end_idx = min(start_idx + batch_size, n_samples)
            batch_X = torch.FloatTensor(features[start_idx:end_idx]).to(device)
            logits = model(batch_X)
            probabilities = F.softmax(logits, dim=1)
            all_predictions.append(probabilities.cpu().numpy())
            
            del batch_X, logits, probabilities
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    predictions = np.vstack(all_predictions)
    
    print(f"  ✅ 预测完成！")
    print(f"    形状: {predictions.shape}")
    print(f"    概率范围: [{predictions.min():.6f}, {predictions.max():.6f}]")
    
    return predictions

def predictions_to_3d_volume(predictions, data_info, include_background=False):
    """
    将预测结果还原为3D softmax volume
    
    Returns:
    --------
    volume_3d : np.ndarray (X, Y, Z, n_classes)
    """
    print(f"\n🔄 还原3D volume...")
    
    original_shape_3d = data_info['original_shape_3d']
    flat_indices = data_info['flat_indices']
    n_classes = predictions.shape[1]
    
    volume_3d = np.zeros((*original_shape_3d, n_classes), dtype=np.float32)
    volume_flat = volume_3d.reshape(-1, n_classes)
    volume_flat[flat_indices] = predictions
    
    if not include_background:
        # 背景区域：class 0概率=1，其他=0
        spatial_mask_3d = data_info['spatial_mask_3d']
        background_indices = np.where(~spatial_mask_3d.flatten())[0]
        volume_flat[background_indices, 0] = 1.0
    
    print(f"  ✅ 3D volume形状: {volume_3d.shape}")
    return volume_3d

def save_prediction_results(volume_3d, data_info, output_dir, model_name, include_background=False):
    """
    保存预测结果（NIfTI + JSON）
    
    Returns:
    --------
    nifti_path, json_path : Path
    """
    print(f"\n💾 保存预测结果...")
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    subject_id = data_info['subject_id']
    bg_str = 'incl' if include_background else 'excl'
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    affine = data_info['affine']
    header = data_info['header'].copy()
    header.set_data_shape(volume_3d.shape)
    header.set_data_dtype(np.float32)
    
    nifti_img = nib.Nifti1Image(volume_3d, affine, header)
    nifti_path = output_dir / f"prediction_3d_{model_name}_{subject_id}_bg_{bg_str}_{timestamp}.nii.gz"
    nib.save(nifti_img, nifti_path)
    
    file_size_mb = nifti_path.stat().st_size / (1024**2)
    print(f"  ✅ NIfTI已保存: {nifti_path.name} ({file_size_mb:.2f} MB)")
    
    # 保存元信息JSON
    info_dict = {
        'model_name': model_name,
        'subject_id': subject_id,
        'timestamp': timestamp,
        'prediction_shape': volume_3d.shape,
        'n_classes': volume_3d.shape[-1],
        'class_labels': STANDARD_LABELS
    }
    
    json_path = output_dir / f"prediction_info_{model_name}_{subject_id}_bg_{bg_str}_{timestamp}.json"
    with open(json_path, 'w') as f:
        json.dump(info_dict, f, indent=2)
    
    print(f"  ✅ JSON已保存: {json_path.name}")
    return nifti_path, json_path

print("✅ 模型和预测函数定义完成！")

In [ ]:
# Cell 5: Error Map可视化函数

def visualize_error_map(predictions, patient_data, model_name, n_slices=5):
    """
    生成error map可视化（使用内存中的预测数据）
    
    Parameters:
    -----------
    predictions : np.ndarray
        模型预测的softmax概率 (n_voxels, n_classes)
    patient_data : dict
        患者数据字典
    model_name : str
        模型名称
    n_slices : int
        每个视图显示的切片数
    """
    print("="*80)
    print(f"ERROR MAP VISUALIZATION - {model_name.upper()} MODEL (IN-MEMORY)")
    print("="*80)
    
    # 1. 获取预测标签 (argmax)
    print("\n📊 Processing predictions...")
    pred_labels_flat = np.argmax(predictions, axis=1).astype(np.int32)
    gt_labels_flat = patient_data['labels'].astype(np.int32)
    
    print(f"  Predicted labels: [{pred_labels_flat.min()}, {pred_labels_flat.max()}]")
    print(f"  Ground truth labels: [{gt_labels_flat.min()}, {gt_labels_flat.max()}]")
    
    # 2. 计算准确率
    print("\n🎨 Creating error map...")
    non_bg_mask = gt_labels_flat > 0
    correct_mask = (pred_labels_flat == gt_labels_flat) & non_bg_mask
    incorrect_mask = (pred_labels_flat != gt_labels_flat) & non_bg_mask
    
    n_correct = correct_mask.sum()
    n_total = non_bg_mask.sum()
    gross_accuracy = n_correct / n_total if n_total > 0 else 0.0
    
    print(f"  Total non-background voxels: {n_total:,}")
    print(f"  Correct predictions: {n_correct:,}")
    print(f"  Incorrect predictions: {incorrect_mask.sum():,}")
    print(f"  ✅ Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)")
    
    # 3. 重建3D error map
    print("\n🔄 Reconstructing 3D error map...")
    original_shape_3d = patient_data['original_shape_3d']
    flat_indices = patient_data['flat_indices']
    
    error_map_3d = np.zeros(original_shape_3d, dtype=np.uint8)
    error_map_flat = error_map_3d.flatten()
    
    # 0=background, 1=correct, 2=incorrect
    error_map_flat[flat_indices[correct_mask]] = 1
    error_map_flat[flat_indices[incorrect_mask]] = 2
    error_map_3d = error_map_flat.reshape(original_shape_3d)
    
    print(f"  Background: {(error_map_3d == 0).sum():,}")
    print(f"  Correct: {(error_map_3d == 1).sum():,}")
    print(f"  Incorrect: {(error_map_3d == 2).sum():,}")
    
    # 4. Per-class分析
    print("\n📊 Per-class accuracy (top 10 by frequency):")
    class_accuracies = {}
    for class_id in np.unique(gt_labels_flat[non_bg_mask]):
        class_mask = gt_labels_flat == class_id
        if class_mask.sum() > 0:
            class_correct = (pred_labels_flat[class_mask] == class_id).sum()
            class_acc = class_correct / class_mask.sum()
            class_accuracies[class_id] = (class_acc, class_mask.sum(), class_correct)
    
    sorted_classes = sorted(class_accuracies.items(), key=lambda x: x[1][1], reverse=True)[:10]
    for class_id, (acc, total, correct) in sorted_classes:
        fs_label = STANDARD_LABELS[class_id]
        print(f"  Class {class_id:2d} (FS={fs_label:3d}): {acc:.3f} ({correct:6,}/{total:6,})")
    
    # 5. 可视化
    print(f"\n🖼️ Generating visualization...")
    
    # 重建GT labels 3D
    gt_labels_3d = np.zeros(original_shape_3d, dtype=np.int32)
    gt_labels_3d_flat = gt_labels_3d.flatten()
    gt_labels_3d_flat[flat_indices] = gt_labels_flat
    gt_labels_3d = gt_labels_3d_flat.reshape(original_shape_3d)
    
    X, Y, Z = original_shape_3d
    z_slices = np.linspace(Z//4, 3*Z//4, n_slices, dtype=int)
    y_slices = np.linspace(Y//4, 3*Y//4, n_slices, dtype=int)
    x_slices = np.linspace(X//4, 3*X//4, n_slices, dtype=int)
    
    colors = ['#2C2C2C', '#00FF00', '#FF0000']  # gray, green, red
    cmap = ListedColormap(colors)
    
    fig, axes = plt.subplots(3, n_slices, figsize=(20, 12))
    fig.suptitle(f'{model_name.upper()} Model - Error Map\\n'
                 f'Gross Accuracy: {gross_accuracy:.4f} ({gross_accuracy*100:.2f}%)\\n'
                 f'Green = Correct | Red = Incorrect | Gray = Background',
                 fontsize=16, fontweight='bold', y=0.98)
    
    # Axial slices
    for i, z_idx in enumerate(z_slices):
        ax = axes[0, i]
        slice_data = error_map_3d[:, :, z_idx].T
        slice_mask = gt_labels_3d[:, :, z_idx] > 0
        slice_acc = (error_map_3d[:, :, z_idx][slice_mask] == 1).sum() / slice_mask.sum() if slice_mask.sum() > 0 else 0
        ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
        ax.set_title(f'Axial Z={z_idx}\\nAcc: {slice_acc:.3f}', fontsize=10)
        ax.axis('off')
    
    # Coronal slices
    for i, y_idx in enumerate(y_slices):
        ax = axes[1, i]
        slice_data = error_map_3d[:, y_idx, :].T
        slice_mask = gt_labels_3d[:, y_idx, :] > 0
        slice_acc = (error_map_3d[:, y_idx, :][slice_mask] == 1).sum() / slice_mask.sum() if slice_mask.sum() > 0 else 0
        ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
        ax.set_title(f'Coronal Y={y_idx}\\nAcc: {slice_acc:.3f}', fontsize=10)
        ax.axis('off')
    
    # Sagittal slices
    for i, x_idx in enumerate(x_slices):
        ax = axes[2, i]
        slice_data = error_map_3d[x_idx, :, :].T
        slice_mask = gt_labels_3d[x_idx, :, :] > 0
        slice_acc = (error_map_3d[x_idx, :, :][slice_mask] == 1).sum() / slice_mask.sum() if slice_mask.sum() > 0 else 0
        ax.imshow(slice_data, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
        ax.set_title(f'Sagittal X={x_idx}\\nAcc: {slice_acc:.3f}', fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 6. 警告
    if gross_accuracy < 0.3:
        print("\n" + "="*80)
        print("⚠️ WARNING: Very low accuracy detected!")
        print("="*80)
        print("Possible causes:")
        print("  1. Label mapping issue (run diagnostic Cell 11)")
        print("  2. Data standardization mismatch (run diagnostic Cell 12)")
        print("  3. Model not generalizing to new data")
        print("="*80)
    
    print("\n✅ Visualization complete!\n")
    return gross_accuracy

print("✅ Error Map可视化函数定义完成！")

## 📥 Part 3: 加载数据

In [ ]:
# Cell 6: 加载新patient数据

patient_data = load_new_patient_data(
    feature_path=NEW_PATIENT_DATA['features'],
    label_path=NEW_PATIENT_DATA['labels'],
    subject_id=NEW_PATIENT_DATA['subject_id'],
    forward_mapping=forward_mapping,
    include_background=PREDICTION_CONFIG['include_background'],
    exclude_features=PREDICTION_CONFIG['exclude_features']
)

print("\n" + "="*80)
print("✅ 新patient数据加载完成！")
print("="*80)

## 🧠 Part 4: KAN模型预测

In [ ]:
# Cell 7: KAN预测 + 保存

print("\n" + "="*80)
print("🚀 KAN模型预测")
print("="*80)

# 加载模型
kan_model, kan_checkpoint = load_trained_model(
    model_path=MODELS['KAN']['model_file'],
    model_type=MODELS['KAN']['model_type'],
    device=device
)

# 预测
kan_predictions = predict_patient(
    model=kan_model,
    data_info=patient_data,
    device=device,
    batch_size=PREDICTION_CONFIG['batch_size']
)

# 还原3D volume
kan_volume_3d = predictions_to_3d_volume(
    predictions=kan_predictions,
    data_info=patient_data,
    include_background=PREDICTION_CONFIG['include_background']
)

# 保存结果
kan_nifti_path, kan_json_path = save_prediction_results(
    volume_3d=kan_volume_3d,
    data_info=patient_data,
    output_dir=MODELS['KAN']['output_dir'],
    model_name='kan',
    include_background=PREDICTION_CONFIG['include_background']
)

# 清理内存 (保留 kan_predictions 供下一个cell使用)
del kan_model, kan_volume_3d
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*80)
print("✅ KAN预测完成！")
print(f"💡 kan_predictions已保留在内存中供error map使用")
print("="*80)

In [ ]:
# Cell 8: KAN Error Map可视化

kan_accuracy = visualize_error_map(
    predictions=kan_predictions,
    patient_data=patient_data,
    model_name='KAN',
    n_slices=5
)

## 🤖 Part 5: DeepMLP模型预测

In [ ]:
# Cell 9: DeepMLP预测 + 保存

print("\n" + "="*80)
print("🚀 DeepMLP模型预测")
print("="*80)

# 加载模型
deepmlp_model, deepmlp_checkpoint = load_trained_model(
    model_path=MODELS['DeepMLP']['model_file'],
    model_type=MODELS['DeepMLP']['model_type'],
    device=device
)

# 预测
deepmlp_predictions = predict_patient(
    model=deepmlp_model,
    data_info=patient_data,
    device=device,
    batch_size=PREDICTION_CONFIG['batch_size']
)

# 还原3D volume
deepmlp_volume_3d = predictions_to_3d_volume(
    predictions=deepmlp_predictions,
    data_info=patient_data,
    include_background=PREDICTION_CONFIG['include_background']
)

# 保存结果
deepmlp_nifti_path, deepmlp_json_path = save_prediction_results(
    volume_3d=deepmlp_volume_3d,
    data_info=patient_data,
    output_dir=MODELS['DeepMLP']['output_dir'],
    model_name='deep_mlp',
    include_background=PREDICTION_CONFIG['include_background']
)

# 清理内存 (保留 deepmlp_predictions 供下一个cell使用)
del deepmlp_model, deepmlp_volume_3d
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*80)
print("✅ DeepMLP预测完成！")
print(f"💡 deepmlp_predictions已保留在内存中供error map使用")
print("="*80)

In [ ]:
# Cell 10: DeepMLP Error Map可视化

deepmlp_accuracy = visualize_error_map(
    predictions=deepmlp_predictions,
    patient_data=patient_data,
    model_name='DeepMLP',
    n_slices=5
)

## 🔍 Part 6: 诊断工具（仅在准确率低时运行）

**如果上面的error map显示准确率很低（<30%），请运行以下诊断cells找出原因。**

In [ ]:
# Cell 11: 标签映射验证 (Label Mapping Verification)
# 运行此cell检查GT labels是否正确映射到0-51

print("="*80)
print("LABEL MAPPING VERIFICATION")
print("="*80)

# 检查 patient_data labels
print("\n📊 Checking patient_data['labels']:")
gt_labels_flat = patient_data['labels'].astype(np.int32)
unique_gt = np.unique(gt_labels_flat)

print(f"  Unique values: {len(unique_gt)}")
print(f"  Range: [{gt_labels_flat.min()}, {gt_labels_flat.max()}]")
print(f"  Values: {sorted(unique_gt.tolist())}")

expected_range = all(0 <= label <= 51 for label in unique_gt)
if expected_range:
    print(f"\n  ✅ Labels are in expected range [0, 51]")
else:
    print(f"\n  ❌ WARNING: Labels are NOT in range [0, 51]!")
    print(f"     This indicates unmapped FreeSurfer labels!")

# 检查 KAN predictions
print("\n📊 Checking KAN predictions:")
pred_labels_flat = np.argmax(kan_predictions, axis=1).astype(np.int32)
unique_pred = np.unique(pred_labels_flat)

print(f"  Unique values: {len(unique_pred)}")
print(f"  Range: [{pred_labels_flat.min()}, {pred_labels_flat.max()}]")
print(f"  Values: {sorted(unique_pred.tolist())}")

# 检查重叠
print("\n📊 Overlap analysis:")
gt_set = set(unique_gt)
pred_set = set(unique_pred)
overlap = gt_set & pred_set

print(f"  Labels in both GT and Pred: {len(overlap)}")
print(f"  Labels only in GT: {len(gt_set - pred_set)}")
print(f"  Labels only in Pred: {len(pred_set - gt_set)}")

# 重新加载原始labels验证
print("\n📂 Re-loading original labels to verify mapping...")
label_img = nib.load(NEW_PATIENT_DATA['labels'])
original_labels = label_img.get_fdata().astype(np.int32).flatten()
original_nonbg = original_labels[original_labels > 0]
unique_original = np.unique(original_nonbg)

print(f"  Original (FreeSurfer) labels: {sorted(unique_original.tolist())}")
print(f"  Mapped (0-51) labels: {sorted(unique_gt.tolist())}")

# 验证映射
print("\n📊 Verifying mapping (first 5 labels):")
all_correct = True
for orig in sorted(unique_original)[:5]:
    if orig in forward_mapping:
        mapped = forward_mapping[orig]
        orig_count = (original_nonbg == orig).sum()
        gt_nonbg = gt_labels_flat[gt_labels_flat > 0]
        mapped_count = (gt_nonbg == mapped).sum()
        match = "✅" if orig_count == mapped_count else "❌"
        print(f"  {match} FreeSurfer {orig:3d} -> Continuous {mapped:2d}: {orig_count:,} voxels")
        if orig_count != mapped_count:
            all_correct = False
    else:
        print(f"  ❌ FreeSurfer {orig:3d} -> NOT IN MAPPING!")
        all_correct = False

# 诊断
print("\n" + "="*80)
print("DIAGNOSIS")
print("="*80)

if expected_range and all_correct:
    print("\n✅ LABEL MAPPING IS CORRECT")
    print("  Low accuracy is NOT due to label mapping.")
    print("  Please run Cell 12 (Data Standardization Diagnostic).")
else:
    print("\n❌ LABEL MAPPING HAS PROBLEMS!")
    print("  This is likely causing the low accuracy.")
    if not expected_range:
        print("\n  Issue: Labels not in range [0-51]")
        print("  Solution: Check forward_mapping in load_new_patient_data()")

print("="*80)

In [ ]:
# Cell 12: 数据标准化诊断 (Data Standardization Diagnostic)
# 运行此cell检查数据标准化是否匹配训练时的方式

print("="*80)
print("DATA STANDARDIZATION DIAGNOSTIC")
print("="*80)

# 当前数据统计
print("\n📊 Current data statistics:")
features = patient_data['features']

print(f"\n  Overall (all voxels, all modalities):")
print(f"    Mean: {features.mean():.6f}")
print(f"    Std:  {features.std():.6f}")
print(f"    Min:  {features.min():.3f}")
print(f"    Max:  {features.max():.3f}")

print(f"\n  Per-modality (first 10):")
n_modalities = features.shape[1]
for i in range(min(10, n_modalities)):
    feat = features[:, i]
    print(f"    Modality {i:2d}: mean={feat.mean():7.4f}, std={feat.std():7.4f}")

# 期望值
print("\n" + "="*80)
print("EXPECTED VALUES (from training)")
print("="*80)
print("\n✅ Patient-wise z-score standardization:")
print("    Overall mean ≈ 0.0")
print("    Overall std  ≈ 1.0")
print("\n  Formula: (features - patient_mean) / patient_std")

# Checkpoint信息
print("\n" + "="*80)
print("CHECKPOINT INFORMATION")
print("="*80)

if 'scalers_info' in kan_checkpoint:
    print("\n📦 Found 'scalers_info' in checkpoint")
    scalers = kan_checkpoint['scalers_info']
    print(f"  Type: {type(scalers)}")
    if isinstance(scalers, dict):
        for k, v in list(scalers.items())[:3]:
            print(f"  {k}: {v}")
else:
    print("\n⚠️  No 'scalers_info' in checkpoint")

# 诊断
print("\n" + "="*80)
print("DIAGNOSIS")
print("="*80)

current_mean = features.mean()
current_std = features.std()

mean_ok = abs(current_mean) < 0.5
std_ok = abs(current_std - 1.0) < 0.5

if mean_ok and std_ok:
    print("\n✅ DATA STANDARDIZATION IS CORRECT")
    print(f"  Mean ≈ 0: {current_mean:.4f}")
    print(f"  Std ≈ 1: {current_std:.4f}")
    print("\n  If accuracy is still low:")
    print("    - Distribution shift (different patient population)")
    print("    - Model overfitting to training data")
    print("    - Different acquisition protocols")
else:
    print("\n❌ DATA STANDARDIZATION MISMATCH!")
    print(f"  Current mean: {current_mean:.4f} (expected ≈ 0.0)")
    print(f"  Current std:  {current_std:.4f} (expected ≈ 1.0)")
    print("\n  This is likely causing the low accuracy!")
    
    print("\n🔧 SOLUTION - Re-standardize the data:")
    print("""
# Run this code to re-standardize:
features_raw = patient_data['features']
patient_mean = features_raw.mean()
patient_std = features_raw.std()

print(f"Before: mean={patient_mean:.6f}, std={patient_std:.6f}")

features_standardized = (features_raw - patient_mean) / patient_std
patient_data['features'] = features_standardized

print(f"After: mean={features_standardized.mean():.6f}, std={features_standardized.std():.6f}")

# Then re-run prediction cells (Cell 7-10)
    """)

print("="*80)

## 📊 Part 7: 结果总结

In [ ]:
# Cell 13: 最终结果总结

print("\n" + "="*80)
print("📊 预测结果总结")
print("="*80)

print(f"\n🎯 Patient: {NEW_PATIENT_DATA['subject_id']}")
print(f"📅 时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📈 预测准确率:")
print(f"  KAN: {kan_accuracy:.4f} ({kan_accuracy*100:.2f}%)")
print(f"  DeepMLP: {deepmlp_accuracy:.4f} ({deepmlp_accuracy*100:.2f}%)")

print(f"\n📁 保存的文件:")
print(f"\n  KAN:")
print(f"    {kan_nifti_path.name} ({kan_nifti_path.stat().st_size / 1024**2:.1f} MB)")
print(f"    {kan_json_path.name}")

print(f"\n  DeepMLP:")
print(f"    {deepmlp_nifti_path.name} ({deepmlp_nifti_path.stat().st_size / 1024**2:.1f} MB)")
print(f"    {deepmlp_json_path.name}")

print(f"\n📍 输出目录:")
print(f"  KAN: {MODELS['KAN']['output_dir']}")
print(f"  DeepMLP: {MODELS['DeepMLP']['output_dir']}")

print("\n" + "="*80)
print("✅ 所有预测完成！")
print("="*80)